# 07 — Diagnosticity of behavioral distance for pruning-mask structure

**Research question (v2).**  Are pairwise *behavioral* distance matrices between
domain-calibrated pruned models diagnostic of *parameter-level* structure — the
overlap of their pruning masks — beyond what the pruning **level** (sparsity) and
the **pruner** already explain?

Each **variant** is one pruned model, keyed by `(pruner, domain, seed, level)`, plus
one shared **baseline** (level 0, unpruned).  Grid (per the v2 spec):

- pruners: `wanda`, `sparsegpt`
- domains (calibration sources): `math` (GSM8K), `mathqa`, `coding` (HumanEval+),
  `mbpp`, `mcq` (ARC-Challenge)
- seeds: 0, 1, 2   |   levels: 10..80 step 10   |   baseline: level 0 (one job)
- eval benchmarks: the same 5 specs, teacher-forced (top-k logprobs).

**What this notebook does.**  For every eval benchmark × distributional metric it
builds a behavioral distance matrix over all *completed* variants, builds a
mask-Jaccard distance matrix from the uploaded mask **digests**, then runs
`pruning_metrics.metrics.cluster_stats` refutation tests:

- **Mantel** — behavioral vs. mask-Jaccard correlation.
- **partial Mantel** — the same, controlling for `|level_i − level_j|` and (separately)
  a same-pruner indicator, to remove the trivial "same sparsity / same pruner ⇒ similar"
  confounds.
- **silhouette / ARI / label-permutation** on calibration-domain labels, overall and
  within `(pruner, level)` strata.

It then contrasts a **domain** pairing (`gsm8k`↔`mathqa`, same math content) with a
**format** pairing (`mathqa`↔`arc`, same MCQ format) in behavioral vs. mask space,
shows illustrative embeddings, and prints a plain-language **verdict**.

The notebook is **tolerant of partial runs**: anything missing is skipped with a
printed `NOTE`, so it runs headlessly end-to-end while the sweep is still in flight.

> Conventions mirror `04_metric_spaces.ipynb` / `05_tsne.ipynb`: caches live under
> `notebooks/experiment/results/`, downloads are skipped when already present, and the
> `per_token.json` schema is consumed unchanged via `pruning_metrics.metrics`.


## 1 · Bootstrap & configuration


In [ ]:
import os
import sys
from pathlib import Path

# Pin to the repo root so the notebook runs from anywhere.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))


In [ ]:
import json
import re

AWS_PROFILE    = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get("RESULTS_BUCKET", "pruning-metrics-results-414266451290")
RESULTS_PREFIX = os.environ.get("PRUNE_EVAL_V2_PREFIX", "prune_eval_v2")

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "experiment_config_v2.json").exists() \
    else (REPO_ROOT / "notebooks" / "experiment")
RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
V2_CACHE_DIR = RESULTS_DIR / "v2_cache"
V2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- distributional metrics (per_token.json consumed unchanged) ----------
METRIC_NAMES = ["kld", "jsd", "emd", "chamfer"]
METRIC_TITLES = {
    "kld": "KLD", "jsd": "JSD", "emd": "EMD", "chamfer": "Chamfer",
}

# ---- grid vocabulary (v2 spec C5) ----------------------------------------
PRUNERS = ["wanda", "sparsegpt"]
# Canonical domain keys and their content/format axes (used by the
# domain-vs-format contrast).  Whatever exact calibration-spec strings P4/P6
# emit are normalised onto these keys by _norm_domain below.
DOMAIN_META = {
    "math":   {"content": "math",    "format": "freeform", "label": "GSM8K (math)"},
    "mathqa": {"content": "math",    "format": "mcq",      "label": "MathQA (math-MCQ)"},
    "coding": {"content": "code",    "format": "freeform", "label": "HumanEval+ (code)"},
    "mbpp":   {"content": "code",    "format": "freeform", "label": "MBPP (code)"},
    "mcq":    {"content": "science", "format": "mcq",      "label": "ARC-Challenge (MCQ)"},
}
DOMAIN_COLORS = {
    "math": "tab:orange", "mathqa": "tab:red", "coding": "tab:green",
    "mbpp": "tab:olive", "mcq": "tab:blue", "baseline": "black",
}
PRUNER_MARKERS = {"wanda": "o", "sparsegpt": "s", "baseline": "*"}


def _norm_domain(raw: str) -> str:
    """Map an arbitrary calibration-spec/domain string onto a canonical key."""
    s = (raw or "").lower()
    if "mathqa" in s or "math_qa" in s:
        return "mathqa"
    if "mbpp" in s:
        return "mbpp"
    if "gsm8k" in s or s == "math" or ":math" in s or s.startswith("math"):
        return "math"
    if "humaneval" in s or "coding" in s or "code" in s:
        return "coding"
    if "arc" in s or "mcq" in s:
        return "mcq"
    return s or "unknown"


# Significance threshold used throughout the refutation tests.
ALPHA = 0.01
N_PERMUTATIONS = int(os.environ.get("V2_PERMUTATIONS", "4999"))

print("Bucket / prefix :", f"s3://{RESULTS_BUCKET}/{RESULTS_PREFIX}/")
print("Cache dir       :", V2_CACHE_DIR)
print("Permutations    :", N_PERMUTATIONS, " alpha =", ALPHA)


### Load the launch manifest

`experiment_config_v2.json` is written incrementally by the orchestration notebook
(`06_prune_eval_v2.ipynb`): one record per launched `(pruner, domain, seed)` job with
its `run_id` and results `uri`.  We read it defensively — the file may be a bare list,
or a dict keyed `launches` / `prune_eval_launches` — and skip malformed rows.


In [ ]:
cfg_path = NOTEBOOK_DIR / "experiment_config_v2.json"
launches = []
if cfg_path.exists():
    _cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if isinstance(_cfg, list):
        raw_launches = _cfg
    else:
        raw_launches = (
            _cfg.get("jobs")
            or _cfg.get("prune_eval_launches")
            or _cfg.get("launches")
            or _cfg.get("prune_eval_v2_launches")
            or []
        )
else:
    raw_launches = []
    print(f"NOTE: {cfg_path.name} not found — no runs to analyse. "
          "Run 06_prune_eval_v2.ipynb first. Downstream cells will no-op gracefully.")

def _run_id_from_uri(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1] if uri else ""

for rec in raw_launches:
    if not isinstance(rec, dict):
        continue
    run_id = rec.get("run_id") or _run_id_from_uri(rec.get("uri", ""))
    uri = rec.get("uri") or (
        f"s3://{RESULTS_BUCKET}/{RESULTS_PREFIX}/{run_id}/" if run_id else ""
    )
    pruner = (rec.get("pruner") or "").lower()
    domain = _norm_domain(rec.get("domain") or rec.get("calibration") or "")
    seed = rec.get("seed")
    if not run_id or pruner not in PRUNERS or seed is None:
        print(f"NOTE: skipping malformed launch record: {rec!r}")
        continue
    launches.append({
        "run_id": run_id, "uri": uri, "pruner": pruner,
        "domain": domain, "seed": int(seed),
        "instance_id": rec.get("instance_id"),
    })

print(f"Launch records: {len(launches)}")
for rec in sorted(launches, key=lambda r: (r["pruner"], r["domain"], r["seed"])):
    print(f"  {rec['pruner']:>9s} | {rec['domain']:>7s} | seed={rec['seed']} | {rec['run_id']}")


## 2 · Sync digests + per-token caches from S3

For every launch we mirror two kinds of object into `results/v2_cache/<run_id>/`:

- `masks/level=NN.digest.npz` — the packed mask **digest** (small; the full masks are
  not needed for Jaccard).
- `level=NN/bench=<spec>/sample=.../per_token.json` — teacher-forced logprobs.

Existing non-empty files are skipped, so re-runs are cheap.  If AWS is unreachable the
cell prints a `NOTE` and continues on whatever is already cached — the notebook stays
runnable offline / on partial data.


In [ ]:
import concurrent.futures

def _split_uri(uri: str) -> tuple[str, str]:
    body = uri[5:]
    bucket, _, key = body.partition("/")
    return bucket, key.rstrip("/")

def _sync_one(s3, rec) -> tuple[int, int]:
    bucket, prefix = _split_uri(rec["uri"])
    local_base = V2_CACHE_DIR / rec["run_id"]
    n_pt = n_dig = 0
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents") or []:
            key = obj["Key"]
            is_pt = key.endswith("per_token.json")
            is_dig = key.endswith(".digest.npz")
            if not (is_pt or is_dig):
                continue
            rel = key[len(prefix):].lstrip("/")
            dest = local_base / rel
            if dest.exists() and dest.stat().st_size > 0:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(bucket, key, str(dest))
            if is_pt:
                n_pt += 1
            else:
                n_dig += 1
    return n_pt, n_dig

if launches:
    try:
        import boto3
        session = boto3.session.Session(profile_name=AWS_PROFILE)
        s3 = session.client("s3")
        print(f"Syncing {len(launches)} run(s) -> {V2_CACHE_DIR} ...")
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
            futs = {pool.submit(_sync_one, s3, rec): rec for rec in launches}
            for fut in concurrent.futures.as_completed(futs):
                rec = futs[fut]
                try:
                    n_pt, n_dig = fut.result()
                    print(f"  {rec['run_id']}: +{n_pt} per_token, +{n_dig} digest")
                except Exception as exc:  # noqa: BLE001
                    print(f"  NOTE {rec['run_id']}: sync error ({exc}) — using cache only")
    except Exception as exc:  # noqa: BLE001
        print(f"NOTE: S3 unreachable ({exc}). Proceeding with whatever is cached.")
else:
    print("No launches — nothing to sync.")


## 3 · Behavioral distance matrices

We scan the local cache into an index and define a **single canonical variant order**
shared by every matrix in the notebook:

- index 0 — the shared **baseline** (level 0), if any level-0 data exists;
- then all `(pruner, domain, seed, level)` variants that have *any* cached data,
  sorted deterministically.

For each `(benchmark, metric)` we build a symmetric matrix `D` where
`D[i, j] = mean over shared tasks of metric(per_token_i, per_token_j)`.  Tasks are keyed
by the `task_id` read from inside each JSON (robust to `_safe_filename` path munging);
teacher-forced sample selection depends only on `(bench, TF_SEED, NUM_TF_SAMPLES)`, so
every variant scores the **same** tasks and rows are positionally comparable.  Matrices
cache to `results/v2_pairwise_<bench>_<metric>.npy` (+ `_meta.json`).


In [ ]:
import numpy as np
from pruning_metrics.metrics import (
    compute_kld, compute_jsd, compute_emd, compute_chamfer,
)

_METRIC_FNS = {
    "kld": compute_kld, "jsd": compute_jsd,
    "emd": compute_emd, "chamfer": compute_chamfer,
}

BASELINE_KEY = "baseline"

def _variant_key(pruner, domain, seed, level):
    if level == 0:
        return BASELINE_KEY
    return f"{pruner}|{domain}|s{seed}|L{level}"

def _run_meta(run_id):
    for rec in launches:
        if rec["run_id"] == run_id:
            return rec
    return None

# ---- scan cache: per_token index + digest paths --------------------------
# tf_index[bench][variant_key][task_id] = Path
# digest_paths[variant_key] = Path (level > 0 only)
# variant_meta[variant_key] = {pruner, domain, seed, level, is_baseline}
tf_index: dict = {}
digest_paths: dict = {}
variant_meta: dict = {}

_level_re = re.compile(r"level=(\d+)")
_bench_re = re.compile(r"bench=([^/]+)")

for rec in launches:
    run_dir = V2_CACHE_DIR / rec["run_id"]
    if not run_dir.exists():
        continue
    # digests: masks/level=NN.digest.npz
    for dig in sorted(run_dir.glob("masks/level=*.digest.npz")):
        m = _level_re.search(dig.name)
        if not m:
            continue
        level = int(m.group(1))
        if level == 0:
            continue
        vk = _variant_key(rec["pruner"], rec["domain"], rec["seed"], level)
        digest_paths[vk] = dig
        variant_meta.setdefault(vk, dict(
            pruner=rec["pruner"], domain=rec["domain"], seed=rec["seed"],
            level=level, is_baseline=False, key=vk,
        ))
    # per_token: level=NN/bench=X/sample=.../per_token.json
    for pt in sorted(run_dir.glob("level=*/bench=*/**/per_token.json")):
        rel = pt.relative_to(run_dir).as_posix()
        lm = _level_re.search(rel)
        bm = _bench_re.search(rel)
        if not (lm and bm):
            continue
        level = int(lm.group(1))
        bench = bm.group(1)
        vk = _variant_key(rec["pruner"], rec["domain"], rec["seed"], level)
        try:
            tid = json.loads(pt.read_text()).get("task_id", pt.parent.name)
        except Exception:  # noqa: BLE001
            continue
        tf_index.setdefault(bench, {}).setdefault(vk, {})[tid] = pt
        if vk == BASELINE_KEY:
            variant_meta.setdefault(vk, dict(
                pruner="baseline", domain="baseline", seed=-1,
                level=0, is_baseline=True, key=vk,
            ))
        else:
            variant_meta.setdefault(vk, dict(
                pruner=rec["pruner"], domain=rec["domain"], seed=rec["seed"],
                level=level, is_baseline=False, key=vk,
            ))

# ---- canonical variant order ---------------------------------------------
def _sort_key(vk):
    m = variant_meta[vk]
    if m["is_baseline"]:
        return (0, "", "", -1, 0)
    return (1, m["pruner"], m["domain"], m["seed"], m["level"])

VARIANTS = sorted(variant_meta.keys(), key=_sort_key)
ROW_META = [variant_meta[vk] for vk in VARIANTS]
VIDX = {vk: i for i, vk in enumerate(VARIANTS)}
BENCHES = sorted(tf_index.keys())

(RESULTS_DIR / "v2_variants.json").write_text(json.dumps(ROW_META, indent=2))
print(f"Variants: {len(VARIANTS)}  (baseline present: {BASELINE_KEY in VIDX})")
print(f"Benchmarks with data: {BENCHES}")
if not VARIANTS:
    print("NOTE: no variants cached — later cells will no-op.")


In [ ]:
import multiprocessing as _mp
import os as _os


def _pairwise_task_worker(args):
    """Compute all 4 metrics for every variant pair present on one task.

    Runs in a forked worker: parses each variant's per_token.json once and
    returns compact (i, j, kld, jsd, emd, chamfer) rows.
    """
    tid, paths = args
    import json as _json
    from pruning_metrics.metrics import (
        compute_chamfer, compute_emd, compute_jsd, compute_kld,
    )
    toks = {}
    for i, p in paths.items():
        try:
            t = _json.loads(open(p).read()).get("per_token", [])
        except Exception:  # noqa: BLE001
            continue
        if t:
            toks[i] = t
    present = sorted(toks)
    rows = []
    for a in range(len(present)):
        i = present[a]
        for b in range(a + 1, len(present)):
            j = present[b]
            rows.append((
                i, j,
                compute_kld(toks[i], toks[j]),
                compute_jsd(toks[i], toks[j]),
                compute_emd(toks[i], toks[j]),
                compute_chamfer(toks[i], toks[j]),
            ))
    return rows


def _pairwise_task_worker_tagged(args):
    """Wrapper returning (tid, rows) so the parent can checkpoint task ids."""
    return args[0], _pairwise_task_worker(args)


def build_behavioral_matrices(bench):
    """All four symmetric (n, n) mean-over-tasks distance matrices for a bench.

    One parallel pass over tasks computes every metric together (a single
    JSON parse per (task, variant)), then caches per metric under the same
    file names / meta-validation scheme as the serial implementation.
    """
    n = len(VARIANTS)
    caches = {
        m: (RESULTS_DIR / f"v2_pairwise_{bench}_{m}.npy",
            RESULTS_DIR / f"v2_pairwise_{bench}_{m}_counts.npy",
            RESULTS_DIR / f"v2_pairwise_{bench}_{m}_meta.json")
        for m in METRIC_NAMES
    }

    # Cache is valid only if every metric matches the current variant order.
    loaded = {}
    for m, (c_npy, c_cnt, c_meta) in caches.items():
        try:
            if c_npy.exists() and c_meta.exists() and                     json.loads(c_meta.read_text()) == VARIANTS:
                D = np.load(c_npy)
                counts = np.load(c_cnt) if c_cnt.exists() else np.zeros((n, n), int)
                if D.shape == (n, n):
                    loaded[m] = (D, counts)
        except Exception:  # noqa: BLE001
            pass
    if len(loaded) == len(METRIC_NAMES):
        return loaded

    bench_idx = tf_index.get(bench, {})
    all_tasks = sorted({t for vk in bench_idx for t in bench_idx[vk]})
    work = []
    for tid in all_tasks:
        paths = {}
        for vk, task_map in bench_idx.items():
            p = task_map.get(tid)
            if p is not None:
                paths[VIDX[vk]] = str(p)
        if len(paths) > 1:
            work.append((tid, paths))

    D4 = {m: np.zeros((n, n), dtype=np.float64) for m in METRIC_NAMES}
    C4 = {m: np.zeros((n, n), dtype=np.int64) for m in METRIC_NAMES}

    # Restart-resilient checkpointing: harness restarts can kill this kernel
    # mid-bench, so partial accumulators are flushed every CKPT_EVERY tasks
    # and reloaded (validated against VARIANTS) on the next run.
    safe_bench = bench.replace("/", "_")
    ckpt_path = RESULTS_DIR / f"v2_ckpt_{safe_bench}.npz"
    done_tasks: set = set()
    if ckpt_path.exists():
        try:
            ck = np.load(ckpt_path, allow_pickle=True)
            if json.loads(str(ck["variants_json"])) == VARIANTS:
                for m in METRIC_NAMES:
                    D4[m] = ck[f"D_{m}"]
                    C4[m] = ck[f"C_{m}"]
                done_tasks = set(ck["done_tasks"].tolist())
                print(f"  {bench}: resuming from checkpoint "
                      f"({len(done_tasks)}/{len(work)} tasks done)")
        except Exception:  # noqa: BLE001
            done_tasks = set()

    work = [(tid, paths) for tid, paths in work if tid not in done_tasks]
    CKPT_EVERY = 25

    def _flush_ckpt():
        tmp = ckpt_path.with_suffix(".tmp.npz")
        payload = {f"D_{m}": D4[m] for m in METRIC_NAMES}
        payload.update({f"C_{m}": C4[m] for m in METRIC_NAMES})
        payload["done_tasks"] = np.array(sorted(done_tasks), dtype=object)
        payload["variants_json"] = np.array(json.dumps(VARIANTS))
        np.savez_compressed(tmp, **payload)
        tmp.replace(ckpt_path)

    workers = max(2, (_os.cpu_count() or 4) - 2)
    ctx = _mp.get_context("fork")
    tids_in_flight = {id(w): w[0] for w in work}
    completed_since_flush = 0
    with ctx.Pool(processes=workers) as pool:
        for tid, rows in pool.imap_unordered(_pairwise_task_worker_tagged, work, chunksize=1):
            for i, j, kld, jsd, emd, chamfer in rows:
                lo, hi = (i, j) if i < j else (j, i)
                for m, v in zip(METRIC_NAMES, (kld, jsd, emd, chamfer)):
                    if np.isfinite(v):
                        D4[m][lo, hi] += v
                        D4[m][hi, lo] += v
                        C4[m][lo, hi] += 1
                        C4[m][hi, lo] += 1
            done_tasks.add(tid)
            completed_since_flush += 1
            if completed_since_flush >= CKPT_EVERY:
                _flush_ckpt()
                completed_since_flush = 0
    if ckpt_path.exists():
        ckpt_path.unlink()

    out = {}
    for m in METRIC_NAMES:
        counts = C4[m]
        with np.errstate(invalid="ignore", divide="ignore"):
            D = np.where(counts > 0, D4[m] / np.where(counts > 0, counts, 1), 0.0)
        c_npy, c_cnt, c_meta = caches[m]
        np.save(c_npy, D)
        np.save(c_cnt, counts)
        c_meta.write_text(json.dumps(VARIANTS, indent=2))
        out[m] = (D, counts)
    return out


def _bench_cost(b):
    """Cheap benches first: MCQ (1-token answers) << code << GSM8K chains."""
    s = b.lower()
    if "arc" in s or "math_qa" in s or "mathqa" in s:
        return 0
    if "humaneval" in s or "mbpp" in s:
        return 1
    return 2


behavioral: dict = {}
for bench in sorted(BENCHES, key=_bench_cost):
    behavioral[bench] = {}
    per_metric = build_behavioral_matrices(bench)
    for metric_name in METRIC_NAMES:
        D, counts = per_metric[metric_name]
        behavioral[bench][metric_name] = D
        n_pairs = int((counts > 0).sum() // 2)
        print(f"  {bench}/{metric_name}: shape={D.shape}, populated pairs={n_pairs}")
if not BENCHES:
    print("NOTE: no benchmarks cached — skipping behavioral matrices.")


## 4 · Mask-Jaccard distance matrix

The mask **digests** (deterministic pseudorandom subsamples of every layer's flat mask,
positionally comparable across variants) give a parameter-space distance via
`pruning_metrics.metrics.masks.jaccard_distance` — `1 − |A∩B| / |A∪B|` over retained
positions.  The baseline (unpruned, no uploaded mask) has no digest, so it is absent from
mask space; an `avail` vector records which canonical rows carry a digest.  Cached to
`results/v2_jaccard.npy`.


In [ ]:
try:
    from pruning_metrics.metrics.masks import load_digest, jaccard_distance
    _MASKS_OK = True
except Exception as exc:  # noqa: BLE001
    _MASKS_OK = False
    print(f"NOTE: masks API unavailable ({exc}); mask-space analyses will be skipped.")

n = len(VARIANTS)
jaccard_D = np.zeros((n, n), dtype=np.float64)
jaccard_avail = np.zeros(n, dtype=bool)

if _MASKS_OK and digest_paths:
    cache_npy = RESULTS_DIR / "v2_jaccard.npy"
    cache_meta = RESULTS_DIR / "v2_jaccard_meta.json"
    reuse = False
    if cache_npy.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get("variants") == VARIANTS:
                jaccard_D = np.load(cache_npy)
                jaccard_avail = np.array(meta["avail"], dtype=bool)
                reuse = jaccard_D.shape == (n, n)
        except Exception:  # noqa: BLE001
            reuse = False
    if not reuse:
        loaded: dict = {}
        for vk, path in digest_paths.items():
            try:
                loaded[vk] = load_digest(path)
                jaccard_avail[VIDX[vk]] = True
            except Exception as exc:  # noqa: BLE001
                print(f"  NOTE: could not load digest for {vk}: {exc}")
        keys = [vk for vk in VARIANTS if vk in loaded]
        for a in range(len(keys)):
            i = VIDX[keys[a]]
            for b in range(a + 1, len(keys)):
                j = VIDX[keys[b]]
                d = float(jaccard_distance(loaded[keys[a]], loaded[keys[b]]))
                jaccard_D[i, j] = jaccard_D[j, i] = d
        np.save(cache_npy, jaccard_D)
        cache_meta.write_text(json.dumps(
            {"variants": VARIANTS, "avail": jaccard_avail.tolist()}, indent=2))
    print(f"Mask-Jaccard: {int(jaccard_avail.sum())} variants with digests.")
else:
    print("NOTE: no digests cached (or masks API unavailable) — mask space empty.")


## 5 · Refutation tests

Using `pruning_metrics.metrics.cluster_stats`, per `(benchmark, metric)`:

1. **Mantel** — correlation of behavioral and mask-Jaccard upper triangles, restricted to
   the variants present in **both** spaces.
2. **partial Mantel** — the same correlation after regressing out a **control** distance:
   (a) `|level_i − level_j|` and (b) a same-pruner indicator (0 if same pruner, else 1).
   A behavioral↔mask link that survives means behavioral distance tracks mask overlap
   *beyond* the trivial sparsity / pruner confounds.
3. **silhouette / ARI / label-permutation** on calibration-**domain** labels in behavioral
   space — overall (all variants) and within `(pruner, level)` strata — asking whether
   models separate by *what they were calibrated on*.


In [ ]:
try:
    from pruning_metrics.metrics.cluster_stats import (
        mantel, partial_mantel, silhouette_by_label,
        ari_vs_labels, label_permutation_pvalue,
    )
    _STATS_OK = True
except Exception as exc:  # noqa: BLE001
    _STATS_OK = False
    print(f"NOTE: cluster_stats API unavailable ({exc}); refutation tests skipped.")


def _submatrix(D, idx):
    idx = np.asarray(idx, dtype=int)
    return D[np.ix_(idx, idx)]

def _level_gap_matrix(idx):
    levels = np.array([ROW_META[i]["level"] for i in idx], dtype=float)
    return np.abs(levels[:, None] - levels[None, :])

def _pruner_mismatch_matrix(idx):
    pr = [ROW_META[i]["pruner"] for i in idx]
    m = np.zeros((len(idx), len(idx)))
    for a in range(len(idx)):
        for b in range(len(idx)):
            m[a, b] = 0.0 if pr[a] == pr[b] else 1.0
    return m

def _behavioral_avail(D):
    """Rows with at least one populated off-diagonal entry."""
    off = D.copy()
    np.fill_diagonal(off, 0.0)
    return np.abs(off).sum(axis=1) > 0


In [ ]:
refutation_rows = []

if _STATS_OK and BENCHES:
    for bench in BENCHES:
        for metric_name in METRIC_NAMES:
            D_beh = behavioral[bench][metric_name]
            beh_ok = _behavioral_avail(D_beh)
            # Variants present in BOTH behavioral and mask space.
            common = np.where(beh_ok & jaccard_avail)[0]
            row = {
                "bench": bench, "metric": metric_name, "n_common": int(common.size),
                "mantel_r": float("nan"), "mantel_p": float("nan"),
                "pmantel_level_r": float("nan"), "pmantel_level_p": float("nan"),
                "pmantel_pruner_r": float("nan"), "pmantel_pruner_p": float("nan"),
            }
            if common.size >= 4:
                Db = _submatrix(D_beh, common)
                Dj = _submatrix(jaccard_D, common)
                try:
                    r, p = mantel(Db, Dj, permutations=N_PERMUTATIONS, seed=0)
                    row["mantel_r"], row["mantel_p"] = float(r), float(p)
                except Exception as exc:  # noqa: BLE001
                    print(f"  NOTE mantel {bench}/{metric_name}: {exc}")
                for ctrl_name, ctrl in (
                    ("level", _level_gap_matrix(common)),
                    ("pruner", _pruner_mismatch_matrix(common)),
                ):
                    try:
                        r, p = partial_mantel(
                            Db, Dj, ctrl, permutations=N_PERMUTATIONS, seed=0)
                        row[f"pmantel_{ctrl_name}_r"] = float(r)
                        row[f"pmantel_{ctrl_name}_p"] = float(p)
                    except Exception as exc:  # noqa: BLE001
                        print(f"  NOTE partial_mantel[{ctrl_name}] "
                              f"{bench}/{metric_name}: {exc}")
            else:
                print(f"  NOTE {bench}/{metric_name}: only {common.size} variant(s) "
                      "in both spaces — need >=4 for Mantel.")
            refutation_rows.append(row)

for r in refutation_rows:
    print(f"  {r['bench']:>16s}/{r['metric']:<7s} "
          f"n={r['n_common']:>2d} "
          f"mantel r={r['mantel_r']:+.3f} p={r['mantel_p']:.4f} | "
          f"partial|level r={r['pmantel_level_r']:+.3f} p={r['pmantel_level_p']:.4f} | "
          f"partial|pruner p={r['pmantel_pruner_p']:.4f}")
if not refutation_rows:
    print("NOTE: no Mantel results (missing data or stats/masks API).")


In [ ]:
# ---- domain-label separation: overall + within (pruner, level) strata -----
def _domain_labels(idx):
    return np.array([ROW_META[i]["domain"] for i in idx])

def _domain_analysis(idx, D):
    """silhouette / ARI / permutation-p for domain labels on a behavioral submatrix."""
    labels = _domain_labels(idx)
    uniq, counts = np.unique(labels, return_counts=True)
    out = {"n": int(idx.size), "n_domains": int(uniq.size),
           "silhouette": float("nan"), "ari": float("nan"),
           "perm_p": float("nan")}
    # silhouette needs 2 <= n_labels <= n_samples - 1.
    if uniq.size < 2 or idx.size < uniq.size + 1:
        return out
    Dsub = _submatrix(D, idx)
    try:
        out["silhouette"] = float(silhouette_by_label(Dsub, labels))
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE silhouette: {exc}")
    try:
        out["ari"] = float(ari_vs_labels(Dsub, labels))
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE ari: {exc}")
    try:
        _stat, out["perm_p"] = label_permutation_pvalue(
            Dsub, labels, stat="silhouette",
            permutations=N_PERMUTATIONS, seed=0)
    except Exception as exc:  # noqa: BLE001
        print(f"    NOTE perm: {exc}")
    return out


domain_rows = []
nonbase = np.array([i for i, m in enumerate(ROW_META) if not m["is_baseline"]], dtype=int)

if _STATS_OK and BENCHES and nonbase.size:
    for bench in BENCHES:
        for metric_name in METRIC_NAMES:
            D = behavioral[bench][metric_name]
            ok = _behavioral_avail(D)
            # Overall
            idx = np.array([i for i in nonbase if ok[i]], dtype=int)
            res = _domain_analysis(idx, D)
            res.update(bench=bench, metric=metric_name, stratum="overall")
            domain_rows.append(res)
            # Within (pruner, level) strata
            strata = {}
            for i in idx:
                strata.setdefault((ROW_META[i]["pruner"], ROW_META[i]["level"]), []).append(i)
            for (pruner, level), members in sorted(strata.items()):
                sub = np.array(members, dtype=int)
                res = _domain_analysis(sub, D)
                res.update(bench=bench, metric=metric_name,
                           stratum=f"{pruner}/L{level}")
                domain_rows.append(res)

_overall = [r for r in domain_rows if r["stratum"] == "overall"]
print("Domain separation (overall, behavioral space):")
for r in _overall:
    print(f"  {r['bench']:>16s}/{r['metric']:<7s} n={r['n']:>2d} "
          f"domains={r['n_domains']} sil={r['silhouette']:+.3f} "
          f"ari={r['ari']:+.3f} perm_p={r['perm_p']:.4f}")
if not domain_rows:
    print("NOTE: no domain-separation results (missing data or stats API).")


## 6 · Domain vs. format

Do models sit closer to same-**content** peers or same-**format** peers?  Contrast a
**domain** pairing `gsm8k`↔`mathqa` (both math content) with a **format** pairing
`mathqa`↔`arc` (both MCQ format), as *mean cross-group distance* in behavioral space and
in mask space.  Smaller ⇒ closer.  If behavioral distance is driven by content, the
domain pairing is closer; if by surface format, the format pairing is.


In [ ]:
def _group_idx(domain, avail):
    return np.array(
        [i for i, m in enumerate(ROW_META)
         if not m["is_baseline"] and m["domain"] == domain and avail[i]],
        dtype=int)

def _mean_cross_group(D, idx_a, idx_b):
    if idx_a.size == 0 or idx_b.size == 0:
        return float("nan")
    block = D[np.ix_(idx_a, idx_b)]
    return float(np.mean(block))

# Pairings named in the spec (generalise via DOMAIN_META content/format axes).
PAIRINGS = [
    ("domain (math content)", "math",   "mathqa"),
    ("format (MCQ surface)",  "mathqa", "mcq"),
]

dvf_rows = []
if BENCHES:
    for bench in BENCHES:
        # behavioral: use JSD as the representative metric (bounded, symmetric),
        # falling back to the first available metric.
        metric_name = "jsd" if "jsd" in behavioral[bench] else METRIC_NAMES[0]
        Dbeh = behavioral[bench][metric_name]
        beh_ok = _behavioral_avail(Dbeh)
        for label, da, db in PAIRINGS:
            ia, ib = _group_idx(da, beh_ok), _group_idx(db, beh_ok)
            dvf_rows.append({
                "bench": bench, "space": f"behavioral/{metric_name}",
                "pairing": label, "domain_a": da, "domain_b": db,
                "mean_dist": _mean_cross_group(Dbeh, ia, ib),
                "n_a": int(ia.size), "n_b": int(ib.size),
            })
# mask space (bench-independent)
if _MASKS_OK and jaccard_avail.any():
    for label, da, db in PAIRINGS:
        ia, ib = _group_idx(da, jaccard_avail), _group_idx(db, jaccard_avail)
        dvf_rows.append({
            "bench": "(mask space)", "space": "mask-jaccard",
            "pairing": label, "domain_a": da, "domain_b": db,
            "mean_dist": _mean_cross_group(jaccard_D, ia, ib),
            "n_a": int(ia.size), "n_b": int(ib.size),
        })

print(f"{'space':>22s} {'bench':>16s} {'pairing':>22s} "
      f"{'mean_dist':>10s}  n_a n_b")
for r in dvf_rows:
    md = r["mean_dist"]
    md_s = "  nan  " if md != md else f"{md:10.4f}"
    print(f"{r['space']:>22s} {r['bench']:>16s} {r['pairing']:>22s} "
          f"{md_s}  {r['n_a']:>3d} {r['n_b']:>3d}")
if not dvf_rows:
    print("NOTE: no domain-vs-format comparison (missing data).")


## 7 · Illustrative embeddings

**Illustration only** — not a statistical test.  We embed one representative
`(benchmark, metric)` behavioral matrix into 2-D (classical MDS via Torgerson
double-centering, then t-SNE, exactly as in `05_tsne.ipynb`) and colour the same points
three ways — by calibration domain, by pruning level, and by pruner — to eyeball which
axis organises the space.


In [ ]:
import matplotlib.pyplot as plt

def classical_mds_coords(D: np.ndarray) -> np.ndarray:
    """Torgerson double-centering (classical MDS / PCoA); pads to >= 2 dims."""
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J
    evals, evecs = np.linalg.eigh(B)
    order = np.argsort(evals)[::-1]
    evals, evecs = evals[order], evecs[:, order]
    keep = evals > max(evals.max(), 0.0) * 1e-9
    X = evecs[:, keep] * np.sqrt(evals[keep])
    if X.shape[1] < 2:
        X = np.hstack([X, np.zeros((n, 2 - X.shape[1]))])
    return X

def _embed_2d(D, idx):
    X = classical_mds_coords(_submatrix(D, idx))
    if idx.size >= 5:
        try:
            from sklearn.manifold import TSNE
            perp = max(2, min(10, idx.size - 1))
            return TSNE(n_components=2, perplexity=perp, init="pca",
                        random_state=42).fit_transform(X)
        except Exception as exc:  # noqa: BLE001
            print(f"  NOTE: t-SNE unavailable ({exc}); using MDS coords.")
    return X[:, :2]

# Pick a representative (bench, metric) with the most usable points.
best = None
for bench in BENCHES:
    for metric_name in METRIC_NAMES:
        D = behavioral[bench][metric_name]
        idx = np.where(_behavioral_avail(D))[0]
        if best is None or idx.size > best[2].size:
            best = (bench, metric_name, idx, D)

if best and best[2].size >= 4:
    bench, metric_name, idx, D = best
    emb = _embed_2d(D, idx)
    meta = [ROW_META[i] for i in idx]

    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    fig.suptitle(
        f"Illustrative embedding — {bench} / {METRIC_TITLES.get(metric_name, metric_name)}"
        f"  (n={idx.size} variants; illustration only, not a test)",
        fontsize=12, fontweight="bold", y=1.02,
    )

    # (a) by domain
    ax = axes[0]
    for i, m in enumerate(meta):
        ax.scatter(emb[i, 0], emb[i, 1],
                   c=DOMAIN_COLORS.get(m["domain"], "gray"),
                   marker="*" if m["is_baseline"] else "o",
                   s=180 if m["is_baseline"] else 80,
                   edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title("by calibration domain", fontsize=10, fontweight="bold")
    seen = {m["domain"] for m in meta}
    ax.legend(handles=[
        plt.Line2D([], [], marker="o", linestyle="None",
                   markerfacecolor=DOMAIN_COLORS.get(d, "gray"),
                   markeredgecolor="k", label=d)
        for d in sorted(seen)], fontsize=8, loc="best")

    # (b) by level
    ax = axes[1]
    levels = np.array([m["level"] for m in meta], dtype=float)
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=levels, cmap="viridis",
                    s=90, edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title("by pruning level", fontsize=10, fontweight="bold")
    fig.colorbar(sc, ax=ax, fraction=0.046, label="level (%)")

    # (c) by pruner
    ax = axes[2]
    pruner_color = {"wanda": "tab:purple", "sparsegpt": "tab:cyan", "baseline": "black"}
    for i, m in enumerate(meta):
        ax.scatter(emb[i, 0], emb[i, 1],
                   c=pruner_color.get(m["pruner"], "gray"),
                   marker=PRUNER_MARKERS.get(m["pruner"], "o"),
                   s=180 if m["is_baseline"] else 80,
                   edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title("by pruner", fontsize=10, fontweight="bold")
    ax.legend(handles=[
        plt.Line2D([], [], marker=PRUNER_MARKERS.get(p, "o"), linestyle="None",
                   markerfacecolor=pruner_color.get(p, "gray"),
                   markeredgecolor="k", label=p)
        for p in sorted({m["pruner"] for m in meta})], fontsize=8, loc="best")

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    EMB_DIR = RESULTS_DIR / "v2_embeddings"
    EMB_DIR.mkdir(exist_ok=True)
    out = EMB_DIR / f"embedding_{bench}_{metric_name}.png"
    fig.savefig(out, dpi=130, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.show()
else:
    print("NOTE: not enough variants for an illustrative embedding (need >=4).")


## 8 · Verdict

The summary table gives, per `(benchmark, metric)`: Mantel `r`/`p`, partial-Mantel
`r`/`p` (controlling `|Δlevel|`), silhouette, ARI, and the domain permutation `p`.

**Decision rule.**  The diagnosticity claim — *behavioral distance is diagnostic of
pruning-mask structure beyond sparsity, and models separate by calibration domain* — is
judged **supported** when partial-Mantel `p < 0.01` for a **majority** of `(bench, metric)`
combos **and** the overall domain-silhouette permutation `p < 0.01`; **refuted** when
neither holds; **mixed** otherwise.


In [ ]:
# Join refutation (Mantel/partial) with the overall domain-separation results.
_dom_overall = {
    (r["bench"], r["metric"]): r
    for r in domain_rows if r["stratum"] == "overall"
}

summary = []
for r in refutation_rows:
    key = (r["bench"], r["metric"])
    dom = _dom_overall.get(key, {})
    summary.append({
        **r,
        "silhouette": dom.get("silhouette", float("nan")),
        "ari": dom.get("ari", float("nan")),
        "domain_perm_p": dom.get("perm_p", float("nan")),
    })

# Header
cols = f"{'bench':>16s} {'metric':>7s} {'n':>3s} {'mantel_r':>9s} {'mantel_p':>9s} " \
       f"{'pM|lvl_r':>9s} {'pM|lvl_p':>9s} {'silhou':>7s} {'ari':>7s} {'dom_p':>7s}"
print(cols)
print("-" * len(cols))

def _f(x, w=9, p=3):
    return (" " * (w - 3) + "nan") if x != x else f"{x:{w}.{p}f}"

for s in summary:
    print(f"{s['bench']:>16s} {s['metric']:>7s} {s['n_common']:>3d} "
          f"{_f(s['mantel_r'])} {_f(s['mantel_p'],9,4)} "
          f"{_f(s['pmantel_level_r'])} {_f(s['pmantel_level_p'],9,4)} "
          f"{_f(s['silhouette'],7)} {_f(s['ari'],7)} {_f(s['domain_perm_p'],7,4)}")

# ---- decision rule -------------------------------------------------------
# A combo only counts as supporting diagnosticity if the partial-Mantel
# correlation is BOTH significant and POSITIVE: partial_mantel returns a
# two-sided |r| p-value, and a significantly negative r (behavioral distance
# anti-tracking mask distance) is evidence against the claim, not for it.
partial_rp = [
    (s["pmantel_level_r"], s["pmantel_level_p"])
    for s in summary
    if s["pmantel_level_p"] == s["pmantel_level_p"]
]
partial_ps = [p for _r, p in partial_rp]
n_partial_sig = sum(1 for r, p in partial_rp if p < ALPHA and r > 0)
majority_partial_sig = bool(partial_rp) and n_partial_sig > len(partial_rp) / 2

dom_ps = [r["perm_p"] for r in _dom_overall.values() if r["perm_p"] == r["perm_p"]]
n_dom_sig = sum(1 for p in dom_ps if p < ALPHA)
domain_sig = bool(dom_ps) and n_dom_sig > len(dom_ps) / 2

if majority_partial_sig and domain_sig:
    verdict = "SUPPORTED"
elif not majority_partial_sig and not domain_sig:
    verdict = "REFUTED"
else:
    verdict = "MIXED"

print()
print("=" * 72)
print("VERDICT — behavioral distance as a diagnostic of pruning-mask structure")
print("=" * 72)
if not summary:
    print("INSUFFICIENT DATA: no completed (bench, metric) combos yet. Re-run once the")
    print("sweep has produced masks + per_token records for several variants.")
else:
    print(f"  partial-Mantel (control |Delta level|) positive and significant at p<{ALPHA}: "
          f"{n_partial_sig}/{len(partial_ps)} combos "
          f"-> majority = {majority_partial_sig}")
    print(f"  domain-silhouette permutation significant at p<{ALPHA}: "
          f"{n_dom_sig}/{len(dom_ps)} benches -> {domain_sig}")
    print()
    print(f"  ==> The diagnosticity claim is: {verdict}")
    print()
    if verdict == "SUPPORTED":
        print("  Behavioral distance between pruned models tracks pruning-mask overlap")
        print("  even after removing the sparsity-level confound, and models separate by")
        print("  calibration domain: behavioral geometry is diagnostic of parameter-level")
        print("  structure.")
    elif verdict == "REFUTED":
        print("  Once |Delta level| is controlled the behavioral<->mask link vanishes and")
        print("  domains do not separate: behavioral distance reflects sparsity, not the")
        print("  specific pruning mask. The diagnosticity claim is not supported.")
    else:
        print("  Evidence is mixed: one of {partial-Mantel majority, domain separation}")
        print("  holds but not both. See the per-(bench, metric) table above; treat any")
        print("  positive signal as suggestive pending more completed variants.")
print("=" * 72)
